# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
EDGES_TXN = f"{WRITE_DB}.{EMP}_v2_edges_transferred"
NODES_ACCT = f"{WRITE_DB}.{EMP}_v2_nodes_account"
SUBG_NIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"
OWN_TBL  = f"{WRITE_DB}.{EMP}_v2_edges_ownership"
FEATS_CUST = f"{WRITE_DB}.{EMP}_v2_cust_features"

ANCHOR_TRAILING = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}

TEST_ONE_ANCHOR = False
USE_GRAPHFRAMES = True

RARE_BENEF_MAX = int(os.environ.get("FRAUD_GRAPH_RARE_BENEF_MAX", "25"))
LPA_ITERS = int(os.environ.get("FRAUD_GRAPH_LPA_ITERS", "5"))
MAX_NODE_DEGREE = int(os.environ.get("FRAUD_GRAPH_MAX_NODE_DEGREE", "1000"))

K_PCTS = [float(value) for value in os.environ.get("FRAUD_GRAPH_EVALUATION_PCTS", "0.02,0.05").split(",")]

OUT_COMM = f"{WRITE_DB}.{EMP}_v2_comm_bad_rate"
OUT_LIFT = f"{WRITE_DB}.{EMP}_v2_increment_lift"
OUT_CUST_COMM = f"{WRITE_DB}.{EMP}_v2_cust_features_comm"

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

anchors = (["202401"] if TEST_ONE_ANCHOR else list(ANCHOR_TRAILING.keys()))
print("anchors:", anchors, "| LPA_ITERS:", LPA_ITERS, "| USE_GRAPHFRAMES:", USE_GRAPHFRAMES,
      "| shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


In [ ]:
GF_JAR = os.environ.get("GRAPHFRAMES_JAR", "graphframes.jar")
CKPT_DIR = os.environ.get("FRAUD_GRAPH_CHECKPOINT_DIR", "file:///tmp/onboarding_fraud/gf_ckpt")
spark.sparkContext.setCheckpointDir(CKPT_DIR)
if USE_GRAPHFRAMES:
    try:
        spark.sparkContext.addPyFile(GF_JAR)
        from graphframes import GraphFrame
        pci = spark.conf.get("spark.graphx.pregel.checkpointInterval", "<NOT SET>")
        print("GraphFrames enabled | checkpoint dir:", CKPT_DIR, "| pregel.checkpointInterval =", pci)
        if pci == "<NOT SET>":
            print("*** WARNING: pregel.checkpointInterval NOT set — falling back to DataFrame LPA to be safe.")
            print("*** (To use GraphFrames LPA, add .config(\"spark.graphx.pregel.checkpointInterval\",\"2\") to the session cell.)")
            USE_GRAPHFRAMES = False
    except Exception as ge:
        print("GraphFrames import failed — falling back to DataFrame LPA (per-iteration checkpoint):", ge)
        USE_GRAPHFRAMES = False
if not USE_GRAPHFRAMES:
    print("Using the DataFrame LPA port (per-iteration reliable checkpoint) | checkpoint dir:", CKPT_DIR)


In [ ]:
def build_edges(anchor):
    months = ANCHOR_TRAILING[anchor]
    e0 = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months)).select("src", "dst")
          .where(F.col("src") != F.col("dst")).persist(StorageLevel.DISK_ONLY))

    fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi"))
    rare = fanin.where(F.col("fi") <= RARE_BENEF_MAX).select("dst")
    e = e0.join(rare, "dst", "inner").distinct()
    und = (e.select("src", "dst")
           .union(e.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
           .persist(StorageLevel.DISK_ONLY))
    und.count(); e0.unpersist()

    if MAX_NODE_DEGREE and MAX_NODE_DEGREE > 0:
        deg = und.groupBy(F.col("src").alias("nid")).agg(F.count("*").alias("d"))
        hub = deg.where(F.col("d") > MAX_NODE_DEGREE).select("nid").persist(StorageLevel.DISK_ONLY)
        nhub = hub.count()
        if nhub:
            und2 = (und.join(hub.select(F.col("nid").alias("h1")), F.col("src") == F.col("h1"), "left_anti")
                       .join(hub.select(F.col("nid").alias("h2")), F.col("dst") == F.col("h2"), "left_anti")
                       .persist(StorageLevel.DISK_ONLY))
            und2.count(); und.unpersist(); und = und2
        hub.unpersist()
        print("   degree-cap %d -> dropped %d ultra-hub node(s) before LPA" % (MAX_NODE_DEGREE, nhub))
    allnodes = und.select("src").union(und.select(F.col("dst").alias("src"))).distinct().select(F.col("src").alias("id"))
    lab = spark.table(SUBG_NIDX).where(F.col("anchor_month") == anchor)
    seeds = lab.where(F.col("is_known_bad") == 1).select("id").distinct()
    cands = (lab.where((F.col("is_candidate") == 1) & (F.col("is_known_bad") == 0))
             .select("id", F.col("target_fraud").cast("int").alias("target_fraud")))
    return und, allnodes, seeds, cands


In [ ]:
def label_propagation_df(und, allnodes, iters):
    lab = allnodes.select("id", F.col("id").alias("label"))
    for it in range(iters):
        msgs = (und.alias("u").join(lab.alias("l"), F.col("u.src") == F.col("l.id"))
                .select(F.col("u.dst").alias("id"), F.col("l.label")))
        cnt = msgs.groupBy("id", "label").count()
        w = Window.partitionBy("id").orderBy(F.desc("count"), F.asc("label"))
        new = cnt.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).select("id", "label")
        lab = new.checkpoint(eager=True)
    return lab.select("id", F.col("label").alias("comm"))

def community_bad_rate(comm, seeds):

    sc  = comm.join(seeds.select("id"), "id", "inner").groupBy("comm").agg(F.count("*").alias("n_seed"))
    tot = comm.groupBy("comm").agg(F.count("*").alias("n_total"))
    rate = (tot.join(sc, "comm", "left").na.fill(0, ["n_seed"])
            .withColumn("comm_bad_rate", F.col("n_seed") / F.col("n_total")).select("comm", "comm_bad_rate"))
    return comm.join(rate, "comm", "left").na.fill(0.0, ["comm_bad_rate"]).select("id", "comm_bad_rate")


In [ ]:
first_save = True
for a in anchors:
    print("=== community pass:", a, "===")
    und, allnodes, seeds, cands = build_edges(a)
    if USE_GRAPHFRAMES:
        g = GraphFrame(allnodes, und)
        comm = g.labelPropagation(maxIter=LPA_ITERS).select("id", F.col("label").alias("comm"))
    else:
        comm = label_propagation_df(und, allnodes, LPA_ITERS)
    cbr = (community_bad_rate(comm, seeds)
           .join(cands.select("id"), "id", "inner")
           .withColumn("anchor_month", F.lit(a))
           .select("id", "anchor_month", "comm_bad_rate"))
    if first_save:
        save_overwrite(cbr, OUT_COMM); first_save = False
    else:
        cbr.write.mode("append").saveAsTable(OUT_COMM)
    print("   saved comm_bad_rate for", a, "->", OUT_COMM)
    und.unpersist()

print("community table:", OUT_COMM, "rows:", spark.table(OUT_COMM).count())


In [ ]:
def capture_at(df, score_col, ks):
    w = Window.orderBy(F.col(score_col).desc_nulls_last())
    ranked = df.select("target_fraud", F.row_number().over(w).alias("rk"))
    tot = df.agg(F.count("*").alias("n"), F.sum("target_fraud").alias("nbad")).first()
    n, nbad = tot["n"], (tot["nbad"] or 0)
    rows = []
    for p in ks:
        K = max(1, int(n * p))
        topbad = ranked.where((F.col("rk") <= K) & (F.col("target_fraud") == 1)).count()
        cap = (topbad / nbad) if nbad else 0.0
        lift = ((topbad / K) / (nbad / n)) if nbad else 0.0
        rows.append((p, K, topbad, round(cap, 4), round(lift, 3)))
    return rows, n, nbad

def pr(df, col):
    return df.withColumn("pr_" + col, F.percent_rank().over(Window.orderBy(F.col(col).asc_nulls_first())))

own = spark.table(OWN_TBL).select(F.col("dst").alias("id"), F.col("src").alias("cust"))
comm_cust = (spark.table(OUT_COMM).join(own, "id", "inner")
             .groupBy("cust", "anchor_month").agg(F.max("comm_bad_rate").alias("comm_bad_rate")))
feats_all = (spark.table(FEATS_CUST).drop("comm_bad_rate")
             .join(comm_cust, ["cust", "anchor_month"], "left").na.fill({"comm_bad_rate": 0.0})
             .persist(StorageLevel.DISK_ONLY))
save_overwrite(feats_all, OUT_CUST_COMM)
print("saved customer features + communities ->", OUT_CUST_COMM)

INCREMENTS = [("baseline", ["pr_in_deg", "pr_out_deg"]),
              ("+multihop", ["pr_closeness_to_bad"]),
              ("+ppr",      ["pr_ppr"]),
              ("+sharedmule", ["pr_n_shared_mule"]),
              ("+community", ["pr_comm_bad_rate"])]
lift_rows = []
for a in anchors:
    fa = feats_all.where(F.col("anchor_month") == a)
    for c in ["in_deg", "out_deg", "closeness_to_bad", "ppr", "n_shared_mule", "comm_bad_rate"]:
        fa = pr(fa, c)
    included = []
    for step, cols in INCREMENTS:
        included = included + cols
        ens = fa.withColumn("ens", sum(F.col(c) for c in included))
        rows, n, nbad = capture_at(ens.select("target_fraud", "ens"), "ens", K_PCTS)
        for (p, K, topbad, cap, lift) in rows:
            lift_rows.append((a, step, p, K, n, nbad, topbad, cap, lift))
            print("   [%s] %-10s top%4.0f%% K=%d  capture=%.3f  lift=%.2f  (bad-cust %d/%d)"
                  % (a, step, p * 100, K, cap, lift, topbad, nbad))

lift_df = spark.createDataFrame(
    lift_rows, ["anchor_month", "step", "top_pct", "K", "n_cust", "n_bad_cust", "bad_in_topK", "capture", "lift"])
save_overwrite(lift_df, OUT_LIFT)
print("refreshed incremental-lift log (CUSTOMER level, with +community):", OUT_LIFT)
spark.table(OUT_LIFT).orderBy("anchor_month", "step", "top_pct").show(60, False)


In [ ]:
SLOPING_FEATURES = ["n_shared_mule", "shares_mule", "ppr", "closeness_to_bad", "comm_bad_rate", "in_deg", "out_deg"]
OUT_SLOPE = f"{WRITE_DB}.{EMP}_v2_feature_sloping"

def _woe_iv(cnt, bad):
    import numpy as np
    good = cnt - bad
    tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6) if tb else bad * 0 + 1e-6
    pg = (good / tg).clip(lower=1e-6) if tg else good * 0 + 1e-6
    return float(((pg - pb) * np.log(pg / pb)).sum())

import numpy as np
import pandas as pd

def _robust_bins(x, max_bins=10):

    s = pd.Series(np.asarray(x, dtype=float))
    nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1:
        return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0)
    out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"),
                     q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values
        return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

pdf = feats_all.select("anchor_month", "target_fraud", *SLOPING_FEATURES).toPandas()
slope_rows = []
for a in sorted(pdf["anchor_month"].unique()):
    d = pdf[pdf["anchor_month"] == a]
    y = d["target_fraud"].astype(int).values
    tot_bad = int(y.sum())
    print("=== sloping @ %s (bad=%d / n=%d) ===" % (a, tot_bad, len(d)))
    for f in SLOPING_FEATURES:
        x = d[f].astype(float)
        try:
            bb = _robust_bins(x.values, max_bins=10)
            g = pd.DataFrame({"b": bb, "y": y}).groupby("b").agg(cnt=("y", "size"), bad=("y", "sum")).sort_index()
            g["event_rate_pct"] = 100.0 * g["bad"] / g["cnt"]
            g["pct_fraud_captured"] = (100.0 * g["bad"] / tot_bad) if tot_bad else 0.0
            iv = _woe_iv(g["cnt"], g["bad"])
            er = g["event_rate_pct"].values
            mono = bool(np.all(np.diff(er) >= -1e-9) or np.all(np.diff(er) <= 1e-9))
            print("  %-18s IV=%.3f  bins=%d  monotonic=%s" % (f, iv, len(g), mono))
            for bi, r in g.reset_index().iterrows():
                slope_rows.append((a, f, int(r["b"]) if pd.notna(r["b"]) else -1,
                                   int(r["cnt"]), int(r["bad"]), round(float(r["event_rate_pct"]), 4),
                                   round(float(r["pct_fraud_captured"]), 4), round(iv, 4), mono))
        except Exception as fe:
            print("  %-18s sloping skipped: %s" % (f, fe))
if slope_rows:
    sdf = spark.createDataFrame(slope_rows, ["anchor_month", "feature", "bin", "cnt", "bad",
                                             "event_rate_pct", "pct_fraud_captured", "iv", "monotonic"])
    save_overwrite(sdf, OUT_SLOPE)
    print("saved full sloping (all features incl. comm_bad_rate):", OUT_SLOPE)


In [ ]:
print("NOTEBOOK 04b (v2) COMPLETE — community bad-rate computed safely (isolated from PPR).")
print("Tables:", OUT_COMM, "(comm_bad_rate) |", OUT_LIFT, "(lift incl. +community).")
print("06 joins core features (_v2_node_features) + community (_v2_comm_bad_rate) for the final score.")
print("Watch whether +community ADDS capture@5% over +ppr; if not, drop it (incremental-then-select).")


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import IPython.display as _disp
from pyspark.sql import functions as _F

CUST_TBL      = f"{WRITE_DB}.{EMP}_v2_cust_features_comm"
PLOT_FEATURES = ["n_shared_mule", "shares_mule", "ppr", "closeness_to_bad", "comm_bad_rate", "in_deg", "out_deg"]
PLOT_ANCHOR   = "202401"
MIN_BIN_FRAC = float(os.environ.get("FRAUD_GRAPH_MIN_BIN_FRACTION", "0.02"))
MIN_STABLE_CNT = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_COUNT", "30"))
MIN_STABLE_BAD = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_POSITIVES", "3"))
RISKY_ON_LEFT = True

def _wilson_lcb(bad, cnt, z=1.96):
    bad = np.asarray(bad, float); cnt = np.asarray(cnt, float)
    p = np.where(cnt > 0, bad / cnt, 0.0); denom = 1 + z*z/np.maximum(cnt, 1)
    centre = p + z*z/(2*np.maximum(cnt, 1)); margin = z*np.sqrt((p*(1-p) + z*z/(4*np.maximum(cnt,1)))/np.maximum(cnt,1))
    return np.where(cnt > 0, np.clip((centre - margin)/denom, 0, 1), 0.0)

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

def _club_sparse(d, n, min_frac):
    d = d.sort_index().copy()
    while len(d) > 2:
        share = d["cnt"] / n
        if share.min() >= min_frac: break
        j = int(share.values.argmin()); k = j + 1 if j == 0 else j - 1
        lo, hi = min(j, k), max(j, k)
        d.loc[d.index[lo], ["cnt", "bad"]] += d.loc[d.index[hi], ["cnt", "bad"]].values
        d.loc[d.index[lo], "hi"] = max(d.loc[d.index[lo], "hi"], d.loc[d.index[hi], "hi"])
        d.loc[d.index[lo], "lo"] = min(d.loc[d.index[lo], "lo"], d.loc[d.index[hi], "lo"])
        d = d.drop(d.index[hi]).reset_index(drop=True)
    return d

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6); pg = (good / tg).clip(lower=1e-6)
    return float(((pg - pb) * np.log(pg / pb)).sum())

_pdf = spark.table(CUST_TBL).where(_F.col("anchor_month") == PLOT_ANCHOR).select("target_fraud", *PLOT_FEATURES).toPandas()
_y = _pdf["target_fraud"].astype(int).values; _tot_bad = int(_y.sum()); _tot_good = len(_y) - _tot_bad
print("plotting anchor %s | customers=%d | frauds=%d" % (PLOT_ANCHOR, len(_pdf), _tot_bad))

def plot_sloping(feature):
    x = _pdf[feature].astype(float).values
    d = (pd.DataFrame({"b": _robust_bins(x), "x": x, "y": _y}).groupby("b")
         .agg(cnt=("y", "size"), bad=("y", "sum"), lo=("x", "min"), hi=("x", "max")).sort_index())
    d = _club_sparse(d, len(_pdf), MIN_BIN_FRAC)
    d = (d.iloc[::-1] if RISKY_ON_LEFT else d).reset_index(drop=True)
    d["event_rate_pct"] = 100.0 * d["bad"] / d["cnt"]
    cum_bad = d["bad"].cumsum(); cum_good = (d["cnt"] - d["bad"]).cumsum()
    d["pct_captured"] = 100.0 * cum_bad / max(_tot_bad, 1)
    ks = float((100.0 * cum_bad / max(_tot_bad, 1) - 100.0 * cum_good / max(_tot_good, 1)).abs().max())
    iv = _woe_iv(d["cnt"], d["bad"])
    d["er_lcb_pct"] = 100.0 * _wilson_lcb(d["bad"].values, d["cnt"].values)
    d["stable"] = (d["cnt"] >= MIN_STABLE_CNT) & (d["bad"] >= MIN_STABLE_BAD)
    labels = [("%.3g" % lo) if lo == hi else ("[%.3g, %.3g]" % (lo, hi)) for lo, hi in zip(d["lo"], d["hi"])]
    xs = range(len(d))
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax1.bar(xs, d["event_rate_pct"], color="#c97b84", alpha=0.85, label="Event Rate (%)")
    for i, v in enumerate(d["event_rate_pct"]): ax1.text(i, v, "%.2f" % v, ha="center", va="bottom", fontsize=8)
    ax1.set_ylabel("Event Rate (%)"); ax1.set_xticks(list(xs)); ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    for i, st in enumerate(d["stable"].values):
        if not st: ax1.patches[i].set_hatch("///"); ax1.patches[i].set_edgecolor("#7a2a34")
    ax2 = ax1.twinx()
    ax2.plot(xs, d["pct_captured"], color="#a23b4e", marker="o", label="Cumulative % Fraud Captured")
    ax2.set_ylabel("Cumulative % Fraud Captured"); ax2.set_ylim(0, 105)
    ax1.set_title("%s @ %s   (IV=%.3f, KS=%.1f)   hatched = UNSTABLE (n<%d)\nEvent Rate (bars) & Cumulative %% Fraud Captured (line)"
                  % (feature, PLOT_ANCHOR, iv, ks, MIN_STABLE_CNT), fontsize=12)
    ax1.legend(loc="upper left"); ax2.legend(loc="lower right"); fig.tight_layout(); plt.show()
    _disp.display(pd.DataFrame({"bin": labels, "# frauds": d["bad"].astype(int).values,
                                "# customers": d["cnt"].astype(int).values,
                                "event_rate_%": d["event_rate_pct"].round(3).values,
                                "event_rate_LCB_%": d["er_lcb_pct"].round(2).values,
                                "cum_%_captured": d["pct_captured"].round(2).values,
                                "stable": d["stable"].values}))

for _f in PLOT_FEATURES:
    try: plot_sloping(_f)
    except Exception as _e: print("  %s: skipped (%s)" % (_f, _e))
